In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(
    root = r"C:\data",
    train = True,
    download = True,
    transform = transform
)

test_dataset = datasets.MNIST(
    root = r"C:\data",
    train = False,
    download = True,
    transform = transform
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True         # Shuffle training data each epoch
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False        # No need to shuffle test data
)

print(f"Training samples : {len(train_dataset)}")   # 60,000
print(f"Test samples     : {len(test_dataset)}")     # 10,000
print(f"Image shape      : {train_dataset[0][0].shape}")  # torch.Size([1, 28, 28])
print(f"Number of classes: {len(train_dataset.classes)}")

Training samples : 60000
Test samples     : 10000
Image shape      : torch.Size([1, 28, 28])
Number of classes: 10


In [21]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        
        self.fc_layers = nn.Sequential(
            nn.Linear(3*3*128, 256),
            nn.ReLU(),
            
            nn.Linear(256, 10)
        )
        
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        
        return x 

        

In [22]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

epochs = 5

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in train_loader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(train_loader)}")

epoch=1/5 & loss=0.13388008605934548
epoch=2/5 & loss=0.03938921082692632
epoch=3/5 & loss=0.028109096156903355
epoch=4/5 & loss=0.022285599809480246
epoch=5/5 & loss=0.018294177181355436


In [24]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels/total_labels*100}")

accuracy = 98.96000000000001
